# Global EM Implied Monetary Policy Pricer

A consistent, meeting-by-meeting implied monetary policy pricer for EM (and CE3/Turkey) rates —
built to replace reliance on Bloomberg MIPR by making every underlying assumption visible and
editable, and by pricing to actual central bank meeting dates instead of fixed horizons.

**Coverage:** Mexico, Chile, Colombia (Latam) · Poland, Hungary, Turkey (CE3) · India, Indonesia,
Thailand, Malaysia, Philippines (Asia). See the *Methodology & glossary* section at the bottom of
this notebook, and the per-country **Data confidence** badge in the app, for how solid each
market's tickers/curve/calendar are — some of the Asia markets run Live-BQL-only on tickers
identified from Bloomberg's standard numbering convention rather than desk confirmation, and should
be confirmed before being traded off.

**How to use this notebook:** click **Run All**. The dashboard below is one continuous page per
country — pick a country from the dropdown, read the market-implied policy path, then use the
scenario section to enter your own central-bank view and see the fair-value gap it implies versus
the market. Assumptions, calibration diagnostics and the full methodology write-up are in the
collapsible **Details** section at the bottom so they don't clutter the main view, but every number
in them is editable.

*Users: research (read the implied path and the "why"), sales (the one-line summaries under Fair
Value), trading (the DV01/P&L panel) — all read the same page, no separate views to reconcile.*

---


In [ ]:
# =============================================================================
# SETUP
# =============================================================================
# Run All to launch the tool. It needs a live BQuant/BQL kernel for "Live BQL"
# mode; outside BQuant it still runs in "Snapshot" mode for the countries that
# have an offline snapshot (Mexico, Chile, Turkey, Poland).

from __future__ import annotations

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from scipy.interpolate import PchipInterpolator, interp1d
from scipy.optimize import least_squares

import plotly.graph_objects as go
from plotly.subplots import make_subplots

import ipywidgets as widgets
from IPython.display import HTML, Markdown, clear_output, display

try:
    import bql
    BQ = bql.Service()
    HAS_BQL = True
except Exception:
    bql = None
    BQ = None
    HAS_BQL = False

if not HAS_BQL:
    display(HTML(
        "<div style='border:1px solid #b45309;background:#78350f;color:#fef3c7;"
        "padding:10px 14px;border-radius:8px;'>"
        "<b>No BQL session detected.</b> This notebook will run in Snapshot-only "
        "mode using the offline curve snapshots baked into the country config "
        "cell below. Run it inside BQuant with a live BQL kernel for live data, "
        "live meeting-calendar refresh and every country in the dropdown."
        "</div>"
    ))


In [ ]:
# =============================================================================
# 1. COUNTRY CONFIGURATION
# =============================================================================
# One row per market. This is the single place that encodes "which instruments,
# which interpolation, which day count, which calendar" for each country -- the
# exact transparency gap the brief calls out in Bloomberg MIPR.
#
# `quotes` = an offline snapshot curve (used by the "Snapshot" data source, and
# as the safe default the notebook falls back to if Live BQL is unavailable).
# `curve_ticker` = a Bloomberg curve object (YCSW0xxx Index) pulled live via
# bq.univ.curvemembers(); when set and `quotes` is empty, "Live BQL" is the
# only usable data source for that country.
#
# DATA CONFIDENCE. Every country below is annotated in `notes` with how solid
# its tickers/curve/calendar are. Mexico, Chile, Colombia, Turkey and Hungary
# carry desk-supplied snapshots and tickers. Poland's snapshot is bootstrapped
# from a supplied WIBOR FRA strip (see PolandCurveBootstrap below). India,
# Indonesia, Thailand, Malaysia and Philippines have no supplied snapshot --
# they run Live BQL only, against curve objects and policy tickers identified
# from the standard Bloomberg numbering convention and public central-bank
# sources rather than desk confirmation. Treat their output as a working
# prototype and confirm tickers with the local rates desk before trading off
# it. Africa has no data at all yet; see AFRICA_EXTENSION_NOTE at the bottom
# of this cell for how to add a market.

from __future__ import annotations

from dataclasses import dataclass
from typing import Dict, Optional, Tuple

import numpy as np
import pandas as pd


@dataclass(frozen=True)
class CountryConfig:
    code: str
    name: str
    region: str
    currency: str
    instrument_name: str
    curve_name: str
    as_of_date: pd.Timestamp
    policy_rate: float
    effective_overnight_rate: float
    overnight_policy_basis_bps: float
    standard_move_bps: float
    day_count_basis: int
    spot_lag_days: int
    policy_ticker: Optional[str]
    overnight_ticker: Optional[str]
    meeting_dates: Tuple[pd.Timestamp, ...]
    quotes: Tuple[Tuple[str, str, float, int], ...]
    calibration_tenors: Tuple[str, ...]
    validation_tenors: Tuple[str, ...]
    trade_tenors: Tuple[str, ...]
    move_lower_bound_bps: float
    move_upper_bound_bps: float
    data_confidence: str
    calendar_confidence: str
    notes: str
    curve_ticker: Optional[str] = None
    # Central-bank meeting calendar lookup, for the Live BQL calendar refresh.
    country_universe: Optional[str] = None
    rate_decision_keywords: Tuple[str, ...] = ("rate decision", "policy rate", "overnight rate", "target rate")
    exclude_keywords: Tuple[str, ...] = ("minutes", "speech", "conference", "forecast", "report")
    fallback_meeting_gap_days: int = 35


def _bootstrap_poland_fra_snapshot() -> Tuple[Tuple[str, str, float, int], ...]:
    """
    Poland has no quoted OIS curve in the supplied context -- only a WIBOR 3M
    FRA strip (context_folder/'Copy of FRAs (needs improving).xlsx'):

        3M spot = 3.8035, then 3x6/6x9/.../21x24 FRAs, each a forward 3-month
        rate for the period starting N months out.

    Each FRA leg is chained into a discount factor (ACT/360, 30-day months,
    consistent with the rest of this notebook's short-end conventions), then
    each cumulative discount factor is converted back into an equivalent
    bullet par rate so it can be fed into the same ShortRateCurve.build() used
    for every other country. This is a simplification (real WIBOR FRA
    settlement/day-count conventions should be confirmed with the Poland desk)
    but it is a real, supplied market curve rather than an invented one.
    """
    fra_legs_pct = {
        3: 3.8035, 6: 3.8100, 9: 3.8250, 12: 3.8475,
        15: 3.8050, 18: 3.7900, 21: 3.7850, 24: 3.7900,
    }
    basis = 360.0
    df = 1.0
    prev_month = 0
    rows = []
    for month, rate_pct in fra_legs_pct.items():
        leg_days = (month - prev_month) * 30
        alpha_leg = leg_days / basis
        df /= (1.0 + (rate_pct / 100.0) * alpha_leg)
        prev_month = month

        total_days = month * 30
        total_alpha = total_days / basis
        par_rate = ((1.0 - df) / (total_alpha * df)) * 100.0

        label = f"{month}M" if month < 12 else ("1Y" if month == 12 else f"{month}M")
        ticker = f"PLN_FRA_STRIP_{month}M_BOOTSTRAP"
        rows.append((label, ticker, round(par_rate, 4), total_days))
    return tuple(rows)


COUNTRIES: Dict[str, CountryConfig] = {

    # ------------------------------------------------------------------ LATAM
    "Mexico": CountryConfig(
        code="MX", name="Mexico", region="Latam", currency="MXN",
        instrument_name="TIIE-F RFR OIS", curve_name="MXN TIIE-F RFR",
        as_of_date=pd.Timestamp("2026-06-27"),
        policy_rate=6.50, effective_overnight_rate=6.50,
        overnight_policy_basis_bps=0.0, standard_move_bps=25.0,
        day_count_basis=360, spot_lag_days=2,
        policy_ticker="MXONBR Index", overnight_ticker="MXONBR Index",
        meeting_dates=tuple(pd.to_datetime([
            "2026-08-06", "2026-09-24", "2026-11-05", "2026-12-17",
        ])),
        quotes=(
            ("1M", "MPSWFA BGN Curncy", 6.5250, 30),
            ("2M", "MPSWFB BGN Curncy", 6.5375, 60),
            ("3M", "MPSWFC BGN Curncy", 6.5450, 91),
            ("6M", "MPSWFF BGN Curncy", 6.5800, 182),
            ("9M", "MPSWFI BGN Curncy", 6.6290, 273),
            ("1Y", "MPSWF1A BGN Curncy", 6.7000, 364),
        ),
        calibration_tenors=("1M", "2M", "3M", "6M"),
        validation_tenors=("9M", "1Y"),
        trade_tenors=("1M", "2M", "3M", "6M"),
        move_lower_bound_bps=-100.0, move_upper_bound_bps=100.0,
        data_confidence="High -- desk-supplied curve snapshot and confirmed tickers.",
        calendar_confidence="High -- Banxico publishes its full-year meeting calendar.",
        notes=(
            "Uses the MXN TIIE-F RFR curve (successor to legacy 28-day TIIE). "
            "1Y tenor is quoted on a 364-day convention (28-day Banxico cycle), "
            "12 days short of a calendar year -- confirmed as a desk basis point to watch."
        ),
        country_universe="MX Country",
        rate_decision_keywords=("overnight rate", "rate decision", "policy rate", "target rate"),
    ),
    "Chile": CountryConfig(
        code="CL", name="Chile", region="Latam", currency="CLP",
        instrument_name="Camara OIS", curve_name="CLP vs Camara",
        as_of_date=pd.Timestamp("2026-06-30"),
        policy_rate=4.50, effective_overnight_rate=4.50,
        overnight_policy_basis_bps=0.0, standard_move_bps=25.0,
        day_count_basis=360, spot_lag_days=0,
        policy_ticker="CHOVCHOV Index", overnight_ticker="ECSRCL Index",
        meeting_dates=tuple(pd.to_datetime([
            "2026-07-28", "2026-09-08", "2026-10-27", "2026-12-15",
        ])),
        quotes=(
            ("1M", "CHSWNIA BGN Curncy", 4.4950, 30),
            ("2M", "CHSWNIB BGN Curncy", 4.5000, 60),
            ("3M", "CHSWNIC BGN Curncy", 4.5100, 91),
            ("6M", "CHSWNIF BGN Curncy", 4.5050, 182),
            ("9M", "CHSWNII BGN Curncy", 4.5050, 273),
            ("1Y", "CHSWNI1 BGN Curncy", 4.5200, 360),
        ),
        calibration_tenors=("1M", "2M", "3M", "6M"),
        validation_tenors=("9M", "1Y"),
        trade_tenors=("1M", "2M", "3M", "6M"),
        move_lower_bound_bps=-100.0, move_upper_bound_bps=100.0,
        data_confidence="High -- desk-supplied curve snapshot and confirmed tickers.",
        calendar_confidence="High -- BCCh publishes its full-year meeting calendar.",
        notes=(
            "Chile has no liquid short-dated futures/FRAs, so the curve is built "
            "entirely from Camara OIS swaps, exactly as in the WIRP-Chile example "
            "this notebook was benchmarked against."
        ),
        country_universe="CL Country",
        rate_decision_keywords=("overnight rate", "rate decision", "monetary policy rate", "mpr"),
    ),
    "Colombia": CountryConfig(
        code="CO", name="Colombia", region="Latam", currency="COP",
        instrument_name="IBR OIS", curve_name="COP IBR",
        as_of_date=pd.Timestamp("2026-07-13"),
        policy_rate=9.25, effective_overnight_rate=9.25,
        overnight_policy_basis_bps=0.0, standard_move_bps=50.0,
        day_count_basis=360, spot_lag_days=0,
        policy_ticker="CORRMIN Index", overnight_ticker="COOVIBR Index",
        meeting_dates=tuple(pd.to_datetime([
            "2026-07-31", "2026-09-30", "2026-10-30", "2026-12-18",
        ])),
        quotes=(),
        curve_ticker="YCSW0329 Index",
        calibration_tenors=("1M", "2M", "3M", "6M"),
        validation_tenors=("9M", "1Y"),
        trade_tenors=("1M", "2M", "3M", "6M"),
        move_lower_bound_bps=-300.0, move_upper_bound_bps=300.0,
        data_confidence="Medium -- curve is live-only (YCSW0329 Index curve object), no offline snapshot supplied.",
        calendar_confidence="High -- BanRep's 2026 rate-decision calendar is published (Jan 30, Mar 31, Apr 30, Jun 30, Jul 31, Sep 30, Oct 30, Dec 18).",
        notes=(
            "BanRep has moved in unusually large increments through H1 2026, so "
            "the move-size bounds are set wide -- tighten once the desk confirms "
            "the expected step size for the rest of 2026."
        ),
        country_universe="CO Country",
        rate_decision_keywords=("overnight rate", "rate decision", "intervention rate", "policy rate"),
    ),

    # -------------------------------------------------------------------- CE3
    "Poland": CountryConfig(
        code="PL", name="Poland", region="CE3", currency="PLN",
        instrument_name="WIBOR OIS / FRA strip", curve_name="PLN WIBOR",
        as_of_date=pd.Timestamp("2026-07-10"),
        policy_rate=3.75, effective_overnight_rate=3.75,
        overnight_policy_basis_bps=0.0, standard_move_bps=25.0,
        day_count_basis=360, spot_lag_days=2,
        policy_ticker="POREANN Index", overnight_ticker="POLONIA Index",
        meeting_dates=tuple(pd.to_datetime([
            "2026-09-02", "2026-10-07", "2026-11-04", "2026-12-02",
        ])),
        quotes=_bootstrap_poland_fra_snapshot(),
        curve_ticker="YCSW0327 Index",
        calibration_tenors=("3M", "6M", "9M", "1Y"),
        validation_tenors=("15M", "18M"),
        trade_tenors=("3M", "6M", "9M", "1Y"),
        move_lower_bound_bps=-100.0, move_upper_bound_bps=100.0,
        data_confidence="Medium -- snapshot is bootstrapped from a supplied WIBOR FRA strip, not a quoted OIS curve; live curve object number is inferred from the desk's numbering notes, not independently confirmed.",
        calendar_confidence="Low-Medium -- July 2026 rate held (4th consecutive hold, per public reporting); NBP typically pauses in August, so Sep/Oct/Nov/Dec dates above are cadence estimates, not the official calendar.",
        notes=(
            "NBP reference rate confirmed at 3.75% as of July 2026. Use Live BQL "
            "calendar refresh before trading off the meeting dates here."
        ),
        country_universe="PL Country",
        rate_decision_keywords=("reference rate", "rate decision", "policy rate"),
        fallback_meeting_gap_days=28,
    ),
    "Hungary": CountryConfig(
        code="HU", name="Hungary", region="CE3", currency="HUF",
        instrument_name="BUBOR 3M OIS/IRS", curve_name="HUF BUBOR 3M",
        as_of_date=pd.Timestamp("2026-07-13"),
        policy_rate=6.00, effective_overnight_rate=6.00,
        overnight_policy_basis_bps=0.0, standard_move_bps=25.0,
        day_count_basis=360, spot_lag_days=2,
        policy_ticker="HBBRDEPO Index", overnight_ticker="BUBORON Index",
        meeting_dates=tuple(pd.to_datetime([
            "2026-07-28", "2026-08-25", "2026-09-22",
            "2026-10-27", "2026-11-24", "2026-12-15",
        ])),
        quotes=(),
        curve_ticker="YCSW0324 Index",
        calibration_tenors=("1M", "2M", "3M", "6M"),
        validation_tenors=("9M", "1Y"),
        trade_tenors=("1M", "2M", "3M", "6M"),
        move_lower_bound_bps=-150.0, move_upper_bound_bps=100.0,
        data_confidence="Medium -- curve is live-only (YCSW0324 Index curve object), no offline snapshot supplied.",
        calendar_confidence="Medium -- only the Mar/Jun/Sep/Dec Inflation Report meetings are officially confirmed by the MNB; the intervening monthly dates are estimated from recent cadence.",
        notes=(
            "The Monetary Council cut the base rate 25 bp to 6.00% on 23 June 2026. "
            "Reconcile the intervening (non quarterly) meeting dates against the "
            "MNB's published Monetary Council calendar before trading off this model."
        ),
        country_universe="HU Country",
        rate_decision_keywords=("base rate", "rate decision", "policy rate"),
    ),
    "Turkey": CountryConfig(
        code="TR", name="Turkey", region="CE3", currency="TRY",
        instrument_name="OIS", curve_name="TRY OIS",
        as_of_date=pd.Timestamp("2026-07-11"),
        policy_rate=37.00, effective_overnight_rate=40.05,
        overnight_policy_basis_bps=305.0, standard_move_bps=250.0,
        day_count_basis=360, spot_lag_days=0,
        policy_ticker="TUBR1WRA Index", overnight_ticker="BISTTREF Index",
        meeting_dates=tuple(pd.to_datetime([
            "2026-07-23", "2026-09-10", "2026-10-22", "2026-12-10",
        ])),
        quotes=(
            ("1W", "TYSO1Z BGN Curncy", 40.049005, 7),
            ("2W", "TYSO2Z BGN Curncy", 39.800000, 14),
            ("1M", "TYSOA BGN Curncy", 39.220000, 30),
            ("2M", "TYSOB BGN Curncy", 39.440000, 60),
            ("3M", "TYSOC BGN Curncy", 39.740000, 91),
            ("6M", "TYSOF BGN Curncy", 38.920000, 182),
            ("9M", "TYSOI BGN Curncy", 38.260000, 273),
            ("1Y", "TYSO1 BGN Curncy", 37.720000, 365),
        ),
        calibration_tenors=("1W", "2W", "1M", "2M", "3M", "6M"),
        validation_tenors=("9M", "1Y"),
        trade_tenors=("1M", "2M", "3M", "6M"),
        move_lower_bound_bps=-500.0, move_upper_bound_bps=250.0,
        data_confidence="High -- desk-supplied curve snapshot and confirmed tickers.",
        calendar_confidence="High -- CBRT publishes its full-year meeting calendar.",
        notes=(
            "Short end (1W-1Y) OIS only, matching the desk's original Turkey "
            "rebuild. 9M/1Y remain validation-only tenors until the 2027 CBRT "
            "calendar is loaded."
        ),
        country_universe="TR Country",
        rate_decision_keywords=("one-week repo rate", "rate decision", "policy rate"),
    ),

    # ------------------------------------------------------------------- ASIA
    "India": CountryConfig(
        code="IN", name="India", region="Asia", currency="INR",
        instrument_name="OIS", curve_name="INR OIS",
        as_of_date=pd.Timestamp("2026-07-10"),
        policy_rate=5.25, effective_overnight_rate=5.25,
        overnight_policy_basis_bps=0.0, standard_move_bps=25.0,
        day_count_basis=365, spot_lag_days=2,
        policy_ticker="INRPYLDP Index", overnight_ticker=None,
        meeting_dates=tuple(pd.to_datetime([
            "2026-08-05", "2026-10-07", "2026-12-04",
        ])),
        quotes=(),
        curve_ticker="YCSW0266 Index",
        calibration_tenors=("1M", "2M", "3M", "6M"),
        validation_tenors=("9M", "1Y"),
        trade_tenors=("1M", "2M", "3M", "6M"),
        move_lower_bound_bps=-100.0, move_upper_bound_bps=100.0,
        data_confidence="Low-Medium -- curve object, policy ticker and current level (5.25%, June 2026) inferred from the desk's curve-number notes and public reporting, not desk-confirmed.",
        calendar_confidence="High -- RBI publishes its bi-monthly MPC calendar; remaining FY26-27 dates confirmed as Aug 5 / Oct 7 / Dec 4, 2026.",
        notes=(
            "RBI meets bi-monthly (6x/year), not monthly -- do not assume a "
            "monthly cadence when extrapolating the calendar."
        ),
        country_universe="IN Country",
        rate_decision_keywords=("repo rate", "rate decision", "policy rate", "mpc"),
    ),
    "Indonesia": CountryConfig(
        code="ID", name="Indonesia", region="Asia", currency="IDR",
        instrument_name="OIS", curve_name="IDR OIS",
        as_of_date=pd.Timestamp("2026-07-10"),
        policy_rate=5.50, effective_overnight_rate=5.50,
        overnight_policy_basis_bps=0.0, standard_move_bps=25.0,
        day_count_basis=360, spot_lag_days=2,
        policy_ticker="IDBIRATE Index", overnight_ticker=None,
        meeting_dates=tuple(pd.to_datetime([
            "2026-07-22", "2026-08-19", "2026-09-23",
            "2026-10-21", "2026-11-18", "2026-12-16",
        ])),
        quotes=(),
        curve_ticker="YCSW0158 Index",
        calibration_tenors=("1M", "2M", "3M", "6M"),
        validation_tenors=("9M", "1Y"),
        trade_tenors=("1M", "2M", "3M", "6M"),
        move_lower_bound_bps=-100.0, move_upper_bound_bps=100.0,
        data_confidence="Low -- curve object explicitly flagged 'confirm?' in the desk's own notes; policy ticker/level (5.50%, after a 50bp hike on 20-May-2026 and a further unscheduled 25bp hike on 9-Jun-2026) taken from public reporting.",
        calendar_confidence="High -- Bank Indonesia publishes its full monthly Board of Governors schedule; decision is announced on the second day of each two-day meeting.",
        notes=(
            "BI has hiked in unusually large, sometimes unscheduled increments in "
            "2026 (rupiah defense) -- treat the move bounds as wide on purpose."
        ),
        country_universe="ID Country",
        rate_decision_keywords=("bi rate", "reverse repo rate", "rate decision", "policy rate"),
    ),
    "Thailand": CountryConfig(
        code="TH", name="Thailand", region="Asia", currency="THB",
        instrument_name="OIS", curve_name="THB OIS",
        as_of_date=pd.Timestamp("2026-07-10"),
        policy_rate=1.00, effective_overnight_rate=1.00,
        overnight_policy_basis_bps=0.0, standard_move_bps=25.0,
        day_count_basis=365, spot_lag_days=2,
        policy_ticker="BTRR1DAY Index", overnight_ticker=None,
        meeting_dates=tuple(pd.to_datetime([
            "2026-08-19", "2026-10-14", "2026-12-09",
        ])),
        quotes=(),
        curve_ticker="YCSW0552 Index",
        calibration_tenors=("1M", "2M", "3M", "6M"),
        validation_tenors=("9M", "1Y"),
        trade_tenors=("1M", "2M", "3M", "6M"),
        move_lower_bound_bps=-75.0, move_upper_bound_bps=75.0,
        data_confidence="Low-Medium -- curve object inferred from the desk's curve-number notes; policy ticker/level (1.00%, held at the 24-Jun-2026 MPC meeting) from public reporting.",
        calendar_confidence="Low -- BOT holds 6 meetings/year; only 29-Apr-2026 and 24-Jun-2026 are confirmed, remaining dates above are a ~56-day cadence estimate, not the official calendar.",
        notes=(
            "Confirm the official 2026 MPC schedule (bot.or.th) before trading off "
            "the estimated meeting dates."
        ),
        country_universe="TH Country",
        rate_decision_keywords=("repurchase rate", "policy rate", "rate decision", "mpc"),
        fallback_meeting_gap_days=56,
    ),
    "Malaysia": CountryConfig(
        code="MY", name="Malaysia", region="Asia", currency="MYR",
        instrument_name="OIS", curve_name="MYR OIS",
        as_of_date=pd.Timestamp("2026-07-10"),
        policy_rate=2.75, effective_overnight_rate=2.75,
        overnight_policy_basis_bps=0.0, standard_move_bps=25.0,
        day_count_basis=365, spot_lag_days=2,
        policy_ticker="MAOPRATE Index", overnight_ticker=None,
        meeting_dates=tuple(pd.to_datetime([
            "2026-09-03", "2026-11-05",
        ])),
        quotes=(),
        curve_ticker="YCSW0209 Index",
        calibration_tenors=("1M", "2M", "3M", "6M"),
        validation_tenors=("9M", "1Y"),
        trade_tenors=("1M", "2M", "3M", "6M"),
        move_lower_bound_bps=-75.0, move_upper_bound_bps=75.0,
        data_confidence="Low-Medium -- curve object inferred from the desk's curve-number notes; policy ticker confirmed (MAOPRATE Index), level 2.75% held since Jul 2025.",
        calendar_confidence="High -- BNM publishes its full-year MPC schedule (6 meetings/year); confirmed 2026 dates.",
        notes="OPR has been held at 2.75% through H1 2026.",
        country_universe="MY Country",
        rate_decision_keywords=("overnight policy rate", "opr", "rate decision", "mpc"),
    ),
    "Philippines": CountryConfig(
        code="PH", name="Philippines", region="Asia", currency="PHP",
        instrument_name="OIS", curve_name="PHP OIS",
        as_of_date=pd.Timestamp("2026-07-10"),
        policy_rate=4.75, effective_overnight_rate=4.75,
        overnight_policy_basis_bps=0.0, standard_move_bps=25.0,
        day_count_basis=360, spot_lag_days=2,
        policy_ticker="PPCBON Index", overnight_ticker=None,
        meeting_dates=tuple(pd.to_datetime([
            "2026-07-30", "2026-09-10", "2026-10-22", "2026-12-03",
        ])),
        quotes=(),
        curve_ticker="YCSW0370 Index",
        calibration_tenors=("1M", "2M", "3M", "6M"),
        validation_tenors=("9M", "1Y"),
        trade_tenors=("1M", "2M", "3M", "6M"),
        move_lower_bound_bps=-100.0, move_upper_bound_bps=100.0,
        data_confidence="Low -- curve object explicitly flagged 'confirm?' in the desk's own notes; policy ticker/level (4.75%, after a 25bp hike on 18-Jun-2026) from public reporting.",
        calendar_confidence="Low -- BSP has met on an irregular, sometimes unscheduled cadence in 2026 (Feb 19, Mar 25 unscheduled, Jun 18); remaining dates above are a ~6-week cadence estimate, not the official calendar.",
        notes=(
            "Confirm the official 2026 Monetary Board schedule (bsp.gov.ph) before "
            "trading off the estimated meeting dates."
        ),
        country_universe="PH Country",
        rate_decision_keywords=("reverse repurchase rate", "rrp", "rate decision", "policy rate"),
        fallback_meeting_gap_days=42,
    ),
}


REGIONS = ["Latam", "CE3", "Asia"]

AFRICA_EXTENSION_NOTE = """
Africa is not yet configured -- the supplied context contains no tickers,
curve numbers or meeting calendars for any African market (the brief lists
Africa only as a future extension, "starting with Latam, CE3, Turkey, and
extending to Asia and Africa"). To add a market (e.g. South Africa), add one
more entry to COUNTRIES above with:
  - a `curve_ticker` (a YCSW0xxx Index curve object) or an offline `quotes`
    snapshot,
  - a `policy_ticker` / `overnight_ticker` for the current policy level,
  - `meeting_dates` for the SARB Monetary Policy Committee,
  - `country_universe` or another calendar lookup for the Live BQL calendar
    refresh.
No other code changes are required -- the curve engine, calibration engine
and dashboard are all fully country-agnostic.
"""

TENOR_DAYS = {
    "1W": 7, "2W": 14, "1M": 30, "2M": 60, "3M": 91,
    "6M": 182, "9M": 273, "1Y": 365,
    "15M": 456, "18M": 547,
}

print(f"Loaded {len(COUNTRIES)} country configurations across {len(REGIONS)} regions.")


In [ ]:
# =============================================================================
# 2. MARKET DATA -- LIVE BQL / OFFLINE SNAPSHOT
# =============================================================================

class MarketData:
    """Loads a short-end curve either from the offline snapshot baked into
    CountryConfig.quotes, or live via BQL (either named tickers or a
    Bloomberg curve object pulled with bq.univ.curvemembers())."""

    @staticmethod
    def snapshot(config: "CountryConfig") -> pd.DataFrame:
        if not config.quotes:
            raise RuntimeError(
                f"No offline snapshot is available for {config.name}. This "
                f"country's curve is only wired up for 'Live BQL' (it reads "
                f"the curve object '{config.curve_ticker}' directly). Switch "
                f"the Data toggle to 'Live BQL' and click Refresh market."
            )
        return pd.DataFrame(
            config.quotes, columns=["tenor", "ticker", "par_rate", "tenor_days"],
        ).assign(source=f"Snapshot {config.as_of_date.date()}")

    @staticmethod
    def _nearest_tenor_label(tenor_days: float, tolerance: float = 0.25):
        best_label, best_gap = None, None
        for label, days in TENOR_DAYS.items():
            gap = abs(tenor_days - days) / float(days)
            if best_gap is None or gap < best_gap:
                best_gap, best_label = gap, label
        return best_label if (best_gap is not None and best_gap <= tolerance) else None

    @staticmethod
    def live_curve_from_curve_object(config: "CountryConfig", service=None) -> pd.DataFrame:
        if bql is None:
            raise RuntimeError("BQL is unavailable in this environment.")
        service = service or BQ

        universe = service.univ.curvemembers(symbols=config.curve_ticker)
        rate_request = bql.Request(universe, {"curve_rate": service.data.curve_rate(side="MID")})
        rate_raw = service.execute(rate_request)[0].df().reset_index()

        id_col = "ID" if "ID" in rate_raw.columns else rate_raw.columns[0]
        rate_col = "curve_rate" if "curve_rate" in rate_raw.columns else rate_raw.columns[-1]

        members = pd.DataFrame({
            "ticker": rate_raw[id_col].astype(str),
            "par_rate": pd.to_numeric(rate_raw[rate_col], errors="coerce"),
        }).dropna(subset=["par_rate"])
        if members.empty:
            raise RuntimeError(f"No curve members/rates returned for {config.curve_ticker}.")

        maturity_request = bql.Request(members["ticker"].tolist(), {"maturity": service.data.maturity()})
        maturity_raw = service.execute(maturity_request)[0].df().reset_index()
        mat_id_col = "ID" if "ID" in maturity_raw.columns else maturity_raw.columns[0]
        mat_col = "maturity" if "maturity" in maturity_raw.columns else maturity_raw.columns[-1]
        maturities = pd.DataFrame({
            "ticker": maturity_raw[mat_id_col].astype(str),
            "maturity": pd.to_datetime(maturity_raw[mat_col], errors="coerce"),
        })

        merged = members.merge(maturities, on="ticker", how="left").dropna(subset=["maturity"])
        merged["tenor_days"] = (merged["maturity"] - config.as_of_date).dt.days
        merged = merged[merged["tenor_days"] > 0].copy()
        merged["tenor"] = merged["tenor_days"].apply(MarketData._nearest_tenor_label)
        merged = merged.dropna(subset=["tenor"])

        merged["_label_days"] = merged["tenor"].map(TENOR_DAYS)
        merged["_gap"] = (merged["tenor_days"] - merged["_label_days"]).abs()
        merged = merged.sort_values("_gap").drop_duplicates("tenor", keep="first")
        merged = merged.sort_values("tenor_days").reset_index(drop=True)
        merged["source"] = f"Live BQL ({config.curve_ticker})"

        result = merged[["tenor", "ticker", "par_rate", "tenor_days", "source"]].copy()
        minimum = max(3, len(config.calibration_tenors))
        if len(result) < minimum:
            raise RuntimeError(
                f"Only {len(result)} usable tenor points came back from "
                f"{config.curve_ticker} after bucketing to standard tenors; "
                f"at least {minimum} are required."
            )
        return result

    @staticmethod
    def live_curve(config: "CountryConfig", service=None) -> pd.DataFrame:
        if bql is None:
            raise RuntimeError("BQL is unavailable in this environment.")
        service = service or BQ

        if config.curve_ticker:
            return MarketData.live_curve_from_curve_object(config, service)

        tickers = [row[1] for row in config.quotes]
        ticker_to_tenor = {row[1]: row[0] for row in config.quotes}
        tenor_to_days = {row[0]: row[3] for row in config.quotes}

        request = bql.Request(tickers, {"par_rate": service.data.px_last()})
        raw = service.execute(request)[0].df().reset_index()
        id_col = "ID" if "ID" in raw.columns else raw.columns[0]
        value_col = "par_rate" if "par_rate" in raw.columns else raw.columns[-1]

        result = pd.DataFrame({
            "ticker": raw[id_col].astype(str),
            "par_rate": pd.to_numeric(raw[value_col], errors="coerce"),
        })
        result["tenor"] = result["ticker"].map(ticker_to_tenor)
        result["tenor_days"] = result["tenor"].map(tenor_to_days)
        result["source"] = "Live BQL"
        result = result.dropna(subset=["tenor", "tenor_days", "par_rate"]).sort_values("tenor_days").reset_index(drop=True)

        minimum = max(3, len(config.calibration_tenors))
        if len(result) < minimum:
            raise RuntimeError(f"BQL returned {len(result)} points; at least {minimum} are required.")
        return result

    @staticmethod
    def live_policy_rate(config: "CountryConfig", service=None) -> float:
        if not config.policy_ticker or bql is None:
            return config.policy_rate
        service = service or BQ
        request = bql.Request([config.policy_ticker], {"policy": service.data.px_last()})
        raw = service.execute(request)[0].df().reset_index()
        value_col = "policy" if "policy" in raw.columns else raw.columns[-1]
        value = pd.to_numeric(raw[value_col], errors="coerce").iloc[0]
        if pd.isna(value):
            raise RuntimeError(f"Policy ticker {config.policy_ticker} returned a non-numeric value.")
        return float(value)

    @staticmethod
    def live_overnight_rate(config: "CountryConfig", service=None) -> float:
        if not config.overnight_ticker or bql is None:
            return config.effective_overnight_rate
        service = service or BQ
        request = bql.Request([config.overnight_ticker], {"on": service.data.px_last()})
        raw = service.execute(request)[0].df().reset_index()
        value_col = "on" if "on" in raw.columns else raw.columns[-1]
        value = pd.to_numeric(raw[value_col], errors="coerce").iloc[0]
        if pd.isna(value):
            return config.effective_overnight_rate
        return float(value)


# =============================================================================
# 3. SHORT-RATE CURVE
# =============================================================================
# Every market quote is treated as a single-payment ("bullet") par instrument
# out to its spot-starting maturity:
#
#     DF(spot, maturity) = 1 / (1 + S * alpha)
#
# Interpolation is applied to LOG DISCOUNT FACTORS, never to raw rates or
# levels. This directly answers Gordian's note that linear interpolation on a
# concave curve underestimates forward rates -- PCHIP (shape-preserving) on
# log-DF avoids that while staying arbitrage-consistent (discount factors
# interpolated this way are always positive and monotonic between nodes).
# A linear-on-log-DF mode is also exposed for users who want to see the
# difference the smoothing choice makes.

class CurveSettings:
    __slots__ = ("day_count_basis", "spot_lag_days", "interpolation_method")

    def __init__(self, day_count_basis: int, spot_lag_days: int, interpolation_method: str = "PCHIP (shape-preserving)"):
        self.day_count_basis = day_count_basis
        self.spot_lag_days = spot_lag_days
        self.interpolation_method = interpolation_method


class ShortRateCurve:
    @staticmethod
    def build(quotes: pd.DataFrame, effective_overnight_rate: float, settings: CurveSettings):
        curve = quotes.copy().sort_values("tenor_days").reset_index(drop=True)
        curve["par_rate"] = pd.to_numeric(curve["par_rate"], errors="coerce")
        curve["tenor_days"] = pd.to_numeric(curve["tenor_days"], errors="coerce")
        curve = curve.dropna(subset=["par_rate", "tenor_days"])
        curve = curve[curve["tenor_days"] > 0].drop_duplicates("tenor_days", keep="last")
        if len(curve) < 3:
            raise ValueError("At least three valid curve quotes are required.")

        spot = settings.spot_lag_days
        curve["start_day"] = spot
        curve["end_day"] = spot + curve["tenor_days"].astype(int)

        df_spot = (1.0 + effective_overnight_rate / 100.0 / settings.day_count_basis) ** (-spot)
        alpha = curve["tenor_days"].to_numpy(float) / settings.day_count_basis
        rates = curve["par_rate"].to_numpy(float) / 100.0
        forward_df = 1.0 / (1.0 + rates * alpha)

        curve["discount_factor"] = df_spot * forward_df
        curve["log_discount_factor"] = np.log(curve["discount_factor"])

        if spot > 0:
            node_days = np.r_[0.0, float(spot), curve["end_day"].to_numpy(float)]
            node_logdfs = np.r_[0.0, np.log(df_spot), curve["log_discount_factor"].to_numpy(float)]
        else:
            node_days = np.r_[0.0, curve["end_day"].to_numpy(float)]
            node_logdfs = np.r_[0.0, curve["log_discount_factor"].to_numpy(float)]

        if settings.interpolation_method.startswith("Linear"):
            logdf_fn = interp1d(node_days, node_logdfs, fill_value="extrapolate")
        else:
            logdf_fn = PchipInterpolator(node_days, node_logdfs, extrapolate=False)
        return curve, logdf_fn

    @staticmethod
    def discount_factor(logdf_fn, day: int) -> float:
        if day <= 0:
            return 1.0
        value = float(logdf_fn(float(day)))
        if not np.isfinite(value):
            raise ValueError(f"Day {day} is outside the calibrated curve.")
        return float(np.exp(value))

    @staticmethod
    def forward_discount_factor(logdf_fn, start_day: int, end_day: int) -> float:
        return ShortRateCurve.discount_factor(logdf_fn, end_day) / ShortRateCurve.discount_factor(logdf_fn, start_day)

    @staticmethod
    def spot_par_rate(logdf_fn, tenor_days: int, settings: CurveSettings) -> float:
        start = settings.spot_lag_days
        end = start + int(tenor_days)
        ratio = ShortRateCurve.forward_discount_factor(logdf_fn, start, end)
        alpha = tenor_days / settings.day_count_basis
        return ((1.0 / ratio - 1.0) / alpha) * 100.0

    @staticmethod
    def rolling_forward_curve(logdf_fn, window_days: int, max_calendar_day: int, settings: CurveSettings) -> pd.DataFrame:
        max_end = settings.spot_lag_days + max_calendar_day
        starts = np.arange(settings.spot_lag_days, max_end - window_days + 1)
        rates = []
        for start in starts:
            end = start + window_days
            ratio = ShortRateCurve.forward_discount_factor(logdf_fn, int(start), int(end))
            rate = ((1.0 / ratio) ** (settings.day_count_basis / window_days) - 1.0) * 100.0
            rates.append(rate)
        return pd.DataFrame({"start_day": starts, "forward_rate": rates})


In [ ]:
# =============================================================================
# 4. MEETING-DATE POLICY-PATH ENGINE
# =============================================================================
# This is the core of the tool and the direct answer to brief item (3):
# "shows implied pricing per central bank meeting date, not just fixed
# horizons." Earlier prototypes (see context_folder) assumed a fixed forward
# window after each meeting (e.g. "28 days after each meeting") -- that is a
# convenient approximation, but it silently mis-prices any meeting that isn't
# exactly one instrument-tenor apart from the next.
#
# Instead: the policy path is modeled as piecewise-constant, jumping only on
# scheduled meeting dates. The jump sizes are the free parameters of a
# nonlinear least-squares fit that reprices every quoted curve instrument
# under that piecewise path and minimizes the repricing error -- i.e. the
# meeting path is *implied by* the curve, not assumed. A handful of small
# regularization penalties (smoothness, size, optional front/back-loading)
# keep the solution well-posed when a single instrument tenor spans more than
# one meeting, which is common information Bloomberg MIPR's fixed-horizon
# design doesn't have to solve for at all.
#
# WIRP-style outputs are read directly off the fitted moves:
#   - "# of standard moves" = fitted move (bp) / standard_move_bps -- this is
#     exactly the Bloomberg WIRP "+0.20 = ~20% chance of a 25bp hike" number.
#   - direction = Hike / Cut / Hold from the sign of the fitted move.

class CalibrationSettings:
    __slots__ = ("smoothness_lambda", "move_size_lambda", "front_load_lambda", "back_load_lambda")

    def __init__(self, smoothness_lambda=0.10, move_size_lambda=0.02, front_load_lambda=0.0, back_load_lambda=0.0):
        self.smoothness_lambda = smoothness_lambda
        self.move_size_lambda = move_size_lambda
        self.front_load_lambda = front_load_lambda
        self.back_load_lambda = back_load_lambda


CALIBRATION_PROFILES = {
    "Balanced": CalibrationSettings(smoothness_lambda=0.10, move_size_lambda=0.02),
    "Smooth (prefer gradual path)": CalibrationSettings(smoothness_lambda=0.45, move_size_lambda=0.04),
    "Front-loaded (moves priced sooner)": CalibrationSettings(smoothness_lambda=0.12, move_size_lambda=0.02, front_load_lambda=0.10),
    "Back-loaded (moves priced later)": CalibrationSettings(smoothness_lambda=0.12, move_size_lambda=0.02, back_load_lambda=0.10),
}

SCENARIO_PRESETS = ["Market-implied", "Hold (no change)", "Dovish (front-loaded cuts)", "Hawkish (front-loaded hikes)", "Custom"]


class PolicyPathEngine:
    @staticmethod
    def boundary_days(config: "CountryConfig", max_curve_day: int) -> np.ndarray:
        meeting_days = [max((date - config.as_of_date).days, 0) for date in config.meeting_dates]
        terminal = max(max_curve_day + config.spot_lag_days + 2, meeting_days[-1] + 32)
        return np.asarray([0] + meeting_days + [terminal], dtype=int)

    @staticmethod
    def accumulation(target_day: int, boundaries: np.ndarray, rates_pct: np.ndarray, config: "CountryConfig") -> float:
        value = 1.0
        for i, rate in enumerate(rates_pct):
            start, end = int(boundaries[i]), int(boundaries[i + 1])
            overlap = max(min(int(target_day), end) - start, 0)
            if overlap > 0:
                value *= (1.0 + float(rate) / 100.0 / config.day_count_basis) ** overlap
            if target_day <= end:
                break
        return value

    @staticmethod
    def forward_df(start_day: int, end_day: int, boundaries: np.ndarray, rates_pct: np.ndarray, config: "CountryConfig") -> float:
        return (
            PolicyPathEngine.accumulation(start_day, boundaries, rates_pct, config)
            / PolicyPathEngine.accumulation(end_day, boundaries, rates_pct, config)
        )

    @staticmethod
    def par_rate(tenor_days: int, boundaries: np.ndarray, rates_pct: np.ndarray, config: "CountryConfig") -> float:
        start = config.spot_lag_days
        end = start + int(tenor_days)
        df = PolicyPathEngine.forward_df(start, end, boundaries, rates_pct, config)
        alpha = tenor_days / config.day_count_basis
        return ((1.0 / df - 1.0) / alpha) * 100.0

    @staticmethod
    def calibrate(config: "CountryConfig", curve: pd.DataFrame, settings: CalibrationSettings = None):
        settings = settings or CalibrationSettings()
        calibration = curve[curve["tenor"].isin(config.calibration_tenors)].copy()
        validation = curve[curve["tenor"].isin(config.validation_tenors)].copy()
        n_meetings = len(config.meeting_dates)
        boundaries = PolicyPathEngine.boundary_days(config, int(curve["tenor_days"].max()))

        weights = 1.0 / np.sqrt(np.maximum(calibration["tenor_days"].to_numpy(float), 7.0) / 7.0)

        longest_rate = float(calibration.sort_values("tenor_days").iloc[-1]["par_rate"])
        approximate_total_move = (longest_rate - config.effective_overnight_rate) * 100.0
        initial_moves = np.repeat(
            np.clip(approximate_total_move / max(n_meetings, 1), config.move_lower_bound_bps, config.move_upper_bound_bps),
            n_meetings,
        )

        def rates_from_moves(move_bps: np.ndarray) -> np.ndarray:
            policy_after = config.policy_rate + np.cumsum(move_bps) / 100.0
            overnight_after = policy_after + config.overnight_policy_basis_bps / 100.0
            return np.r_[config.effective_overnight_rate, overnight_after]

        def residuals(move_bps: np.ndarray) -> np.ndarray:
            rates = rates_from_moves(move_bps)
            errors = []
            for (_, row), weight in zip(calibration.iterrows(), weights):
                model = PolicyPathEngine.par_rate(int(row["tenor_days"]), boundaries, rates, config)
                errors.append((model - float(row["par_rate"])) * 100.0 * weight)

            smooth_penalty = np.sqrt(settings.smoothness_lambda) * np.diff(move_bps) / config.standard_move_bps
            size_penalty = np.sqrt(settings.move_size_lambda) * move_bps / config.standard_move_bps
            front_weights = np.arange(n_meetings, dtype=float)
            back_weights = front_weights[::-1]
            front_penalty = np.sqrt(settings.front_load_lambda) * front_weights * move_bps / config.standard_move_bps
            back_penalty = np.sqrt(settings.back_load_lambda) * back_weights * move_bps / config.standard_move_bps

            return np.r_[errors, smooth_penalty, size_penalty, front_penalty, back_penalty]

        result = least_squares(
            residuals, initial_moves,
            bounds=(np.full(n_meetings, config.move_lower_bound_bps), np.full(n_meetings, config.move_upper_bound_bps)),
            xtol=1e-11, ftol=1e-11, gtol=1e-11, max_nfev=20000,
        )
        if not result.success:
            raise RuntimeError(f"Calibration failed: {result.message}")

        moves = result.x
        rates = rates_from_moves(moves)
        policy_after = config.policy_rate + np.cumsum(moves) / 100.0
        number_of_moves = moves / config.standard_move_bps

        path = pd.DataFrame({
            "meeting_date": config.meeting_dates,
            "move_bps_raw": moves,
            "number_of_standard_moves": number_of_moves,
            "post_policy_rate": policy_after,
            "post_overnight_rate": rates[1:],
            "direction": np.where(moves > 1e-6, "Hike", np.where(moves < -1e-6, "Cut", "Hold")),
        })

        diagnostics = []
        for _, row in pd.concat([calibration, validation]).iterrows():
            model = PolicyPathEngine.par_rate(int(row["tenor_days"]), boundaries, rates, config)
            diagnostics.append({
                "tenor": row["tenor"],
                "use": "Calibration" if row["tenor"] in config.calibration_tenors else "Validation",
                "market_rate": float(row["par_rate"]),
                "model_rate": model,
                "error_bps": (model - float(row["par_rate"])) * 100.0,
            })
        diagnostics = pd.DataFrame(diagnostics)
        calibration_errors = diagnostics.loc[diagnostics["use"] == "Calibration", "error_bps"].to_numpy(float)
        metrics = {
            "rmse_bps": float(np.sqrt(np.mean(calibration_errors ** 2))),
            "max_abs_error_bps": float(np.max(np.abs(calibration_errors))),
            "success": bool(result.success),
        }
        return path, diagnostics, metrics, boundaries, rates

    @staticmethod
    def scenario_path(config: "CountryConfig", market_path: pd.DataFrame, user_moves_bps) -> pd.DataFrame:
        if len(user_moves_bps) != len(market_path):
            raise ValueError("Enter one move per meeting.")
        scenario = market_path.copy()
        scenario["user_move_bps"] = [float(x) for x in user_moves_bps]
        scenario["user_policy_rate"] = config.policy_rate + scenario["user_move_bps"].cumsum() / 100.0
        scenario["user_overnight_rate"] = scenario["user_policy_rate"] + config.overnight_policy_basis_bps / 100.0
        scenario["user_minus_market_bps"] = (scenario["user_policy_rate"] - scenario["post_policy_rate"]) * 100.0
        return scenario

    @staticmethod
    def preset_moves(config: "CountryConfig", market_path: pd.DataFrame, preset: str):
        n = len(market_path)
        if preset == "Market-implied":
            return market_path["move_bps_raw"].tolist()
        if preset == "Hold (no change)":
            return [0.0] * n
        if preset == "Dovish (front-loaded cuts)":
            step = -abs(config.standard_move_bps)
            half = max(1, n // 2)
            return [step] * half + [0.0] * (n - half)
        if preset == "Hawkish (front-loaded hikes)":
            step = abs(config.standard_move_bps)
            half = max(1, n // 2)
            return [step] * half + [0.0] * (n - half)
        return None  # "Custom" -- leave widgets as the user set them

    @staticmethod
    def selected_trade(config: "CountryConfig", logdf_fn, tenor_days: int, boundaries: np.ndarray,
                        scenario_rates: np.ndarray, notional: float, direction: str) -> pd.DataFrame:
        curve_settings = CurveSettings(config.day_count_basis, config.spot_lag_days)
        market_rate = ShortRateCurve.spot_par_rate(logdf_fn, tenor_days, curve_settings)
        scenario_rate = PolicyPathEngine.par_rate(tenor_days, boundaries, scenario_rates, config)

        start = config.spot_lag_days
        end = start + int(tenor_days)
        scenario_df = PolicyPathEngine.forward_df(start, end, boundaries, scenario_rates, config)
        alpha = tenor_days / config.day_count_basis

        fixed_pv = market_rate / 100.0 * alpha * scenario_df * notional
        floating_pv = (1.0 - scenario_df) * notional
        payer_pv = floating_pv - fixed_pv
        position_pv = payer_pv if direction == "Pay fixed" else -payer_pv
        dv01 = alpha * scenario_df * notional / 10000.0
        gap_bps = (scenario_rate - market_rate) * 100.0

        return pd.DataFrame([{
            "market_rate_pct": market_rate,
            "scenario_fair_rate_pct": scenario_rate,
            "scenario_minus_market_bps": gap_bps,
            "direction": direction,
            "notional": notional,
            "dv01_per_bp": dv01,
            "scenario_pv": position_pv,
            "trade_signal": "Pay fixed" if gap_bps > 1.0 else ("Receive fixed" if gap_bps < -1.0 else "Near fair"),
        }])


In [ ]:
# =============================================================================
# 5. MEETING CALENDAR -- LIVE BQL REFRESH
# =============================================================================
# Answers "can we download central bank meeting dates from Bloomberg?" (yes,
# via bq.data.calendar) directly rather than relying only on the static dates
# baked into each CountryConfig. The static list is still useful as a
# fallback for offline/Snapshot use, and its confidence is documented per
# country in CountryConfig.calendar_confidence.

class MeetingCalendar:
    @staticmethod
    def live_meeting_dates(config: "CountryConfig", horizon_days: int = 400, service=None) -> pd.DataFrame:
        if bql is None:
            raise RuntimeError("BQL is unavailable in this environment.")
        if not config.country_universe:
            raise RuntimeError(f"No calendar universe configured for {config.name}.")
        service = service or BQ

        start = config.as_of_date.date()
        end = (config.as_of_date + pd.Timedelta(days=horizon_days)).date()

        data_item = service.data.calendar(
            type="CENTRAL_BANKS", subtype="ALL_CENTRAL_BANKS",
            dates=service.func.range(start.strftime("%Y-%m-%d"), end.strftime("%Y-%m-%d")),
            view="EXTENDED",
        )
        request = bql.Request(config.country_universe, data_item)
        raw = service.execute(request)[0].df().reset_index()

        def find_col(df, candidates):
            lower_map = {str(c).lower(): c for c in df.columns}
            for cand in candidates:
                if cand.lower() in lower_map:
                    return lower_map[cand.lower()]
            for col in df.columns:
                for cand in candidates:
                    if cand.lower() in str(col).lower():
                        return col
            return None

        release_col = find_col(raw, ["RELEASE_DATE", "release_date"])
        name_col = find_col(raw, ["EVENT_NAME", "event_name"])
        if release_col is None or name_col is None:
            raise RuntimeError(f"Unexpected calendar schema: {list(raw.columns)}")

        df = raw.copy()
        df["Meeting Date"] = pd.to_datetime(df[release_col], errors="coerce")
        df["Event Name"] = df[name_col].astype(str)
        df = df.dropna(subset=["Meeting Date"])

        include = [k.lower() for k in config.rate_decision_keywords]
        exclude = [k.lower() for k in config.exclude_keywords]
        event_lower = df["Event Name"].str.lower()
        keep = event_lower.apply(lambda x: any(k in x for k in include)) & ~event_lower.apply(lambda x: any(k in x for k in exclude))
        df = df[keep].drop_duplicates(subset=["Meeting Date"]).sort_values("Meeting Date").reset_index(drop=True)

        df = df[df["Meeting Date"] >= pd.Timestamp(config.as_of_date)]
        if df.empty:
            raise RuntimeError("BQL calendar returned no matching rate-decision events in range.")
        return df[["Meeting Date", "Event Name"]]

    @staticmethod
    def resolve(config: "CountryConfig", use_live: bool) -> "tuple[tuple[pd.Timestamp, ...], str]":
        """Returns (meeting_dates, source_label). Falls back to the static
        config list (with a note) if Live BQL calendar isn't available or fails."""
        if use_live and bql is not None:
            try:
                live = MeetingCalendar.live_meeting_dates(config)
                dates = tuple(pd.to_datetime(live["Meeting Date"]).tolist())
                if dates:
                    return dates, "Live BQL central bank calendar"
            except Exception as exc:
                return config.meeting_dates, f"Static fallback (Live BQL calendar failed: {exc})"
        return config.meeting_dates, f"Static config list -- {config.calendar_confidence}"


In [ ]:
# =============================================================================
# 6. DASHBOARD -- SINGLE SCROLLING VIEW
# =============================================================================
# Deliberately not a multi-tab layout: research, sales and trading should all
# be able to read one page top-to-bottom -- country -> current pricing ->
# meeting-by-meeting chart -> table -> "what if" scenario -> fair value.
# Assumptions, calibration diagnostics and full methodology are collapsed
# into an Accordion beneath the main flow so they're one click away without
# competing for space with the primary view.

from dataclasses import replace


class GlobalEMPolicyPricer:
    def __init__(self):
        self.config = COUNTRIES["Mexico"]
        self.quotes = None
        self.curve = None
        self.logdf = None
        self.path = None
        self.diagnostics = None
        self.metrics = None
        self.boundaries = None
        self.market_rates = None
        self.scenario = None
        self.trade = None
        self.move_widgets = []
        self.calendar_source_label = ""

        self._build_widgets()
        self._build_layout()

    # ------------------------------------------------------------------
    # Widgets
    # ------------------------------------------------------------------
    def _country_options(self):
        ordered = []
        for region in REGIONS:
            for name, cfg in COUNTRIES.items():
                if cfg.region == region:
                    ordered.append((f"{region} — {name}", name))
        return ordered

    def _build_widgets(self):
        c = self.config

        self.country = widgets.Dropdown(
            options=self._country_options(), value="Mexico",
            description="Country:", layout=widgets.Layout(width="260px"),
        )
        self.data_source = widgets.ToggleButtons(
            options=["Snapshot", "Live BQL"], value="Snapshot", description="Curve data:",
        )
        self.calendar_source = widgets.ToggleButtons(
            options=["Static list", "Live BQL"], value="Static list", description="Meeting calendar:",
        )
        self.refresh = widgets.Button(description="Refresh", button_style="info", icon="refresh")

        self.policy_rate = widgets.FloatText(value=c.policy_rate, description="Policy rate %:", layout=widgets.Layout(width="200px"))
        self.overnight = widgets.FloatText(value=c.effective_overnight_rate, description="Effective O/N %:", layout=widgets.Layout(width="215px"))
        self.basis = widgets.FloatText(value=c.overnight_policy_basis_bps, description="O/N-policy bp:", layout=widgets.Layout(width="210px"))
        self.standard_move = widgets.FloatText(value=c.standard_move_bps, description="Standard move bp:", layout=widgets.Layout(width="220px"))
        self.interpolation = widgets.Dropdown(
            options=["PCHIP (shape-preserving)", "Linear"], value="PCHIP (shape-preserving)",
            description="Curve interpolation:", style={"description_width": "140px"}, layout=widgets.Layout(width="320px"),
        )
        self.calibration_method = widgets.Dropdown(
            options=list(CALIBRATION_PROFILES), value="Balanced",
            description="Meeting allocation:", style={"description_width": "140px"}, layout=widgets.Layout(width="320px"),
        )

        self.scenario_preset = widgets.Dropdown(options=SCENARIO_PRESETS, value="Market-implied", description="Scenario preset:", layout=widgets.Layout(width="280px"))
        self.apply_preset = widgets.Button(description="Apply preset", icon="magic")
        self.run_scenario = widgets.Button(description="Run scenario", button_style="success", icon="play")

        self.trade_tenor = widgets.Dropdown(options=list(c.trade_tenors), value=c.trade_tenors[-1], description="Tenor:", layout=widgets.Layout(width="170px"))
        self.direction = widgets.Dropdown(options=["Pay fixed", "Receive fixed"], value="Pay fixed", description="Direction:", layout=widgets.Layout(width="210px"))
        self.notional = widgets.FloatText(value=100_000_000.0, description=f"Notional {c.currency}:", layout=widgets.Layout(width="260px"))

        self.header_out = widgets.Output()
        self.chart_out = widgets.Output()
        self.table_out = widgets.Output()
        self.scenario_controls_out = widgets.Output()
        self.fair_value_out = widgets.Output()
        self.assumptions_out = widgets.Output()
        self.diagnostics_out = widgets.Output()
        self.methodology_out = widgets.Output()

        self.country.observe(self._country_changed, names="value")
        self.refresh.on_click(lambda _: self.run(refresh=True))
        self.apply_preset.on_click(lambda _: self._apply_preset())
        self.run_scenario.on_click(lambda _: self.run(refresh=False))
        for w in (self.calibration_method, self.interpolation):
            w.observe(lambda _: self.run(refresh=True), names="value")

    def _build_layout(self):
        top_controls = widgets.VBox([
            widgets.HTML("<h2 style='margin-bottom:4px;'>Global EM Implied Monetary Policy Pricer</h2>"),
            widgets.HTML(
                "<div style='color:#9ca3af;margin-bottom:8px;'>Meeting-by-meeting implied policy "
                "pricing from live curves — replacing fixed-horizon MIPR-style output. "
                "Pick a country, read the market-implied path, then enter your own view below.</div>"
            ),
            widgets.HBox([self.country, self.data_source, self.calendar_source, self.refresh]),
        ])

        self._assumptions_box = widgets.VBox([self.assumptions_out])
        self._diagnostics_box = widgets.VBox([self.diagnostics_out])
        self._methodology_box = widgets.VBox([self.methodology_out])

        self.accordion = widgets.Accordion(children=[self._assumptions_box, self._diagnostics_box, self._methodology_box])
        self.accordion.set_title(0, "Assumptions & conventions (editable)")
        self.accordion.set_title(1, "Calibration diagnostics (model vs market fit)")
        self.accordion.set_title(2, "Methodology & glossary")
        self.accordion.selected_index = None

        self.page = widgets.VBox([
            top_controls,
            self.header_out,
            self.chart_out,
            self.table_out,
            widgets.HTML("<h3 style='margin-top:18px;'>Your scenario — enter a policy path and reprice</h3>"),
            self.scenario_controls_out,
            self.fair_value_out,
            widgets.HTML("<h3 style='margin-top:18px;'>Details</h3>"),
            self.accordion,
        ])

    def display(self):
        display(self.page)

    # ------------------------------------------------------------------
    # State
    # ------------------------------------------------------------------
    def _country_changed(self, change):
        self.config = COUNTRIES[change["new"]]
        c = self.config
        self.policy_rate.value = c.policy_rate
        self.overnight.value = c.effective_overnight_rate
        self.basis.value = c.overnight_policy_basis_bps
        self.standard_move.value = c.standard_move_bps
        self.trade_tenor.options = list(c.trade_tenors)
        self.trade_tenor.value = c.trade_tenors[-1]
        self.notional.description = f"Notional {c.currency}:"
        self.scenario_preset.value = "Market-implied"

        if not c.quotes and c.curve_ticker:
            self.data_source.value = "Live BQL"
        else:
            self.data_source.value = "Snapshot"
        self.calendar_source.value = "Static list"

        self.quotes = None
        self.run(refresh=True)

    def _editable_config(self):
        return replace(
            self.config,
            policy_rate=float(self.policy_rate.value),
            effective_overnight_rate=float(self.overnight.value),
            overnight_policy_basis_bps=float(self.basis.value),
            standard_move_bps=float(self.standard_move.value),
        )

    def _load_market(self):
        c = self.config
        if self.data_source.value == "Live BQL":
            self.quotes = MarketData.live_curve(c)
            self.policy_rate.value = MarketData.live_policy_rate(c)
            self.overnight.value = MarketData.live_overnight_rate(c)
            self.config = self._editable_config()
        else:
            self.quotes = MarketData.snapshot(c)

        meeting_dates, self.calendar_source_label = MeetingCalendar.resolve(
            self.config, use_live=(self.calendar_source.value == "Live BQL")
        )
        self.config = replace(self.config, meeting_dates=meeting_dates)

    def _build_scenario_controls(self):
        self.move_widgets = [
            widgets.FloatText(value=float(move), description=date.strftime("%d %b %Y"), layout=widgets.Layout(width="230px"))
            for date, move in zip(self.path["meeting_date"], self.path["move_bps_raw"])
        ]
        with self.scenario_controls_out:
            clear_output()
            display(widgets.HTML(
                "<div style='color:#9ca3af;margin-bottom:6px;'>Move at each meeting, in bp. "
                "Negative = cut, positive = hike. Use a preset for a quick start, or type your own.</div>"
            ))
            display(widgets.HBox([self.scenario_preset, self.apply_preset]))
            display(widgets.GridBox(
                children=self.move_widgets,
                layout=widgets.Layout(grid_template_columns="repeat(3, 245px)", grid_gap="4px 16px", margin="8px 0"),
            ))
            display(widgets.HBox([self.trade_tenor, self.direction, self.notional, self.run_scenario]))

    def _apply_preset(self):
        moves = PolicyPathEngine.preset_moves(self.config, self.path, self.scenario_preset.value)
        if moves is not None:
            for widget, move in zip(self.move_widgets, moves):
                widget.value = float(move)
        self.run(refresh=False)

    def _user_moves(self):
        return [float(w.value) for w in self.move_widgets]

    # ------------------------------------------------------------------
    # Model run
    # ------------------------------------------------------------------
    def run(self, refresh=False):
        with self.header_out:
            clear_output()
            display(HTML("<b>Loading market and calibrating...</b>"))
        try:
            self.config = self._editable_config()
            c = self.config

            if refresh or self.quotes is None:
                self._load_market()
                self.config = self._editable_config()
                c = self.config

                curve_settings = CurveSettings(c.day_count_basis, c.spot_lag_days, self.interpolation.value)
                self.curve, self.logdf = ShortRateCurve.build(self.quotes, c.effective_overnight_rate, curve_settings)
                (self.path, self.diagnostics, self.metrics, self.boundaries, self.market_rates) = PolicyPathEngine.calibrate(
                    c, self.curve, CALIBRATION_PROFILES[self.calibration_method.value],
                )
                self._build_scenario_controls()

            self.scenario = PolicyPathEngine.scenario_path(c, self.path, self._user_moves())
            scenario_rates = np.r_[c.effective_overnight_rate, self.scenario["user_overnight_rate"].to_numpy(float)]
            self.trade = PolicyPathEngine.selected_trade(
                c, self.logdf, TENOR_DAYS[self.trade_tenor.value], self.boundaries,
                scenario_rates, float(self.notional.value), self.direction.value,
            )

            self.render_header()
            self.render_chart()
            self.render_table()
            self.render_fair_value()
            self.render_assumptions()
            self.render_diagnostics()
            self.render_methodology()

        except Exception:
            import traceback
            with self.header_out:
                clear_output()
                display(HTML(f"<h3 style='color:#f87171'>Dashboard error</h3><pre>{traceback.format_exc()}</pre>"))

    # ------------------------------------------------------------------
    # Rendering
    # ------------------------------------------------------------------
    def render_header(self):
        c = self.config
        first = self.path.iloc[0]
        total_priced = float(self.path["move_bps_raw"].sum())
        badge_color = {"High": "#14532d", "Medium": "#713f12", "Low-Medium": "#7c2d12", "Low": "#7f1d1d"}
        conf_key = next((k for k in badge_color if c.data_confidence.startswith(k)), "Low")

        cards = [
            ("CURRENT POLICY", f"{c.policy_rate:.2f}%", c.curve_name),
            ("EFFECTIVE O/N", f"{c.effective_overnight_rate:.2f}%", c.instrument_name),
            ("NEXT MEETING", f"{first['move_bps_raw']:+.0f} bp ({first['direction']})", str(first["meeting_date"].date())),
            ("TOTAL PRICED", f"{total_priced:+.0f} bp", f"through {self.path.iloc[-1]['meeting_date'].date()}"),
            ("CURVE FIT", f"{self.metrics['rmse_bps']:.2f} bp RMSE", self.quotes["source"].iloc[0] if "source" in self.quotes else ""),
        ]
        card_html = "".join(
            f"""<div style="border:1px solid #444;border-radius:8px;padding:10px 13px;min-width:165px;flex:1;">
                  <div style="color:#9ca3af;font-size:11px;">{title}</div>
                  <div style="font-size:19px;font-weight:700;margin-top:2px;">{value}</div>
                  <div style="color:#8b949e;font-size:11px;">{sub}</div>
                </div>"""
            for title, value, sub in cards
        )
        with self.header_out:
            clear_output()
            display(HTML(f"""
            <div style="border:1px solid #555;border-radius:10px;padding:14px 16px;margin:4px 0 10px;">
              <div style="display:flex;justify-content:space-between;align-items:baseline;flex-wrap:wrap;">
                <div style="font-size:18px;font-weight:700;">{c.name} · {c.region} · {c.currency}</div>
                <div style="background:{badge_color[conf_key]};color:#f5f5f5;padding:3px 9px;border-radius:10px;font-size:11px;">
                  Data confidence: {c.data_confidence.split(' -- ')[0]}
                </div>
              </div>
              <div style="color:#9ca3af;margin-top:2px;">As of {c.as_of_date.date()} · Meeting calendar: {self.calendar_source_label}</div>
              <div style="display:flex;flex-wrap:wrap;gap:8px;margin-top:12px;">{card_html}</div>
            </div>
            """))

    def _step_coordinates(self, dates, levels, current_level, chart_end):
        x, y = [self.config.as_of_date], [current_level]
        previous = current_level
        for date, level in zip(dates, levels):
            x.extend([date, date])
            y.extend([previous, float(level)])
            previous = float(level)
        x.append(chart_end)
        y.append(previous)
        return x, y

    def render_chart(self):
        c = self.config
        meeting_dates = self.path["meeting_date"].tolist()
        chart_end = meeting_dates[-1] + pd.Timedelta(days=21)

        market_x, market_y = self._step_coordinates(meeting_dates, self.path["post_policy_rate"], c.policy_rate, chart_end)
        scenario_x, scenario_y = self._step_coordinates(meeting_dates, self.scenario["user_policy_rate"], c.policy_rate, chart_end)

        fig = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.68, 0.32], vertical_spacing=0.06)

        fig.add_trace(go.Scatter(
            x=market_x, y=market_y, mode="lines", name="Market-implied policy path",
            line={"width": 3, "color": "#38bdf8"}, hovertemplate="%{x|%d %b %Y}<br>%{y:.3f}%<extra></extra>",
        ), row=1, col=1)
        fig.add_trace(go.Scatter(
            x=meeting_dates, y=self.path["post_policy_rate"], mode="markers",
            marker={"size": 8, "color": "#38bdf8"}, showlegend=False,
            hovertemplate="%{x|%d %b %Y}<br>%{y:.3f}%<extra></extra>",
        ), row=1, col=1)

        moves_differ = not np.allclose(self.scenario["user_move_bps"].to_numpy(float), self.path["move_bps_raw"].to_numpy(float), atol=0.5)
        if moves_differ:
            fig.add_trace(go.Scatter(
                x=scenario_x, y=scenario_y, mode="lines", name="Your scenario",
                line={"width": 2, "color": "#f472b6", "dash": "dash"},
                hovertemplate="%{x|%d %b %Y}<br>%{y:.3f}%<extra></extra>",
            ), row=1, col=1)

        fig.add_trace(go.Bar(
            x=meeting_dates, y=self.path["number_of_standard_moves"], name=f"# of {c.standard_move_bps:.0f}bp moves priced",
            marker={"color": ["#f97316" if v >= 0 else "#38bdf8" for v in self.path["number_of_standard_moves"]]},
            hovertemplate="%{x|%d %b %Y}<br>%{y:+.2f} moves<extra></extra>",
        ), row=2, col=1)

        fig.add_vline(x=c.as_of_date.strftime("%Y-%m-%d"), line_width=1, line_dash="dot", line_color="#9ca3af", row=1, col=1)
        fig.add_vline(x=c.as_of_date.strftime("%Y-%m-%d"), line_width=1, line_dash="dot", line_color="#9ca3af", row=2, col=1)

        fig.update_yaxes(title_text="Implied policy rate (%)", row=1, col=1)
        fig.update_yaxes(title_text="# moves", row=2, col=1)
        fig.update_layout(
            height=560, hovermode="x unified", template="plotly_dark",
            legend={"orientation": "h", "yanchor": "bottom", "y": 1.02, "x": 0.0},
            margin=dict(t=40, r=10, l=10, b=10),
            title=f"{c.name}: implied {c.curve_name} policy path by meeting date",
        )
        with self.chart_out:
            clear_output()
            display(fig)

    def render_table(self):
        table = self.path[["meeting_date", "move_bps_raw", "number_of_standard_moves", "direction", "post_policy_rate"]].copy()
        table.columns = ["Meeting", "Implied move (bp)", "# standard moves", "Direction", "Implied policy rate (%)"]
        with self.table_out:
            clear_output()
            display(HTML("<b>Market-implied meeting path</b>"))
            display(table.round(3).reset_index(drop=True))

    def render_fair_value(self):
        t = self.trade.iloc[0]
        with self.fair_value_out:
            clear_output()
            display(HTML(f"""
            <div style="border:1px solid #444;border-radius:8px;padding:12px 14px;margin-top:6px;">
              <b>Fair value — {self.direction.value} {self.trade_tenor.value} {self.config.instrument_name},
              notional {self.config.currency} {self.notional.value:,.0f}</b><br>
              Market rate: <b>{t['market_rate_pct']:.3f}%</b> &nbsp;|&nbsp;
              Your scenario fair rate: <b>{t['scenario_fair_rate_pct']:.3f}%</b> &nbsp;|&nbsp;
              Gap: <b>{t['scenario_minus_market_bps']:+.1f} bp</b> &nbsp;|&nbsp;
              Signal: <b>{t['trade_signal']}</b><br>
              DV01: {self.config.currency} {t['dv01_per_bp']:,.0f}/bp &nbsp;|&nbsp;
              Scenario P&amp;L on this position: <b>{self.config.currency} {t['scenario_pv']:,.0f}</b>
              <div style="color:#9ca3af;font-size:12px;margin-top:8px;">
                Research: the curve implies {self.path['move_bps_raw'].sum():+.0f} bp of net moves through
                {self.path.iloc[-1]['meeting_date'].date()}. Sales: if a client's view matches your scenario
                above, this tenor is the cleanest way to express the gap. Trading: {t['trade_signal']} is
                the P&amp;L-maximizing side of this trade under your scenario, before transaction costs.
              </div>
            </div>
            """))

    def render_assumptions(self):
        c = self.config
        rows = pd.DataFrame({
            "Assumption": [
                "Country / curve source", "Instrument", "Calendar convention", "Spot lag",
                "Curve interpolation", "Meeting allocation (regularization)", "O/N – policy basis",
                "Standard move size", "Meeting calendar source", "Calendar confidence", "Curve data confidence",
            ],
            "Value": [
                f"{c.name} — {c.curve_name} ({c.instrument_name})",
                c.instrument_name,
                f"ACT/{c.day_count_basis}",
                f"{c.spot_lag_days} business day(s)",
                self.interpolation.value,
                self.calibration_method.value,
                f"{c.overnight_policy_basis_bps:+.1f} bp",
                f"{c.standard_move_bps:.0f} bp",
                self.calendar_source_label,
                c.calendar_confidence,
                c.data_confidence,
            ],
        })
        with self.assumptions_out:
            clear_output()
            display(HTML(
                "<div style='color:#9ca3af;margin-bottom:6px;'>Every number below is editable in the "
                "controls above (policy rate, effective O/N, basis, standard move, interpolation, "
                "meeting allocation, data source) — nothing here is hidden inside the math.</div>"
            ))
            display(rows)
            display(HTML(f"<div style='color:#9ca3af;font-size:12px;margin-top:6px;'>{c.notes}</div>"))

    def render_diagnostics(self):
        with self.diagnostics_out:
            clear_output()
            display(HTML(
                f"<b>Model fit:</b> RMSE {self.metrics['rmse_bps']:.2f} bp, "
                f"max abs error {self.metrics['max_abs_error_bps']:.2f} bp across calibration tenors."
            ))
            display(self.diagnostics.round(4))

    def render_methodology(self):
        with self.methodology_out:
            clear_output()
            display(Markdown(METHODOLOGY_TEXT))


In [ ]:
# =============================================================================
# 7. METHODOLOGY TEXT (used by the in-app accordion)
# =============================================================================

METHODOLOGY_TEXT = r"""
## Methodology & glossary

*(This same text is available inline in the app under "Details -> Methodology & glossary".)*

### What this replaces, and why

Bloomberg MIPR shows an implied policy rate per country at fixed horizons (e.g. "Brazil 1y implied = x%")
without disclosing which instruments it used, which interpolation, which day count, whether policy is
assumed to move only at meetings, or how the first fixing is handled. This tool makes every one of those
choices explicit and editable, and prices to actual **meeting dates** rather than fixed horizons.

### 1. Instrument choice per market

EM short-rate curves are swap/OIS-based almost everywhere in this tool's scope, because that is what is
liquid: Chile has no liquid short-dated futures or FRAs, so its curve is built entirely from Camara OIS;
Mexico trades on the TIIE-F RFR swap curve; Turkey, Colombia and Hungary similarly price off OIS/IRS
curves. Poland is the one exception in this build: no OIS curve was supplied, so its snapshot is
bootstrapped from a quoted WIBOR **FRA** strip (3x6, 6x9, ... 21x24) — see "Poland's FRA bootstrap" below.
Bonds were not used anywhere in this build: government bond yields mix a credit/liquidity premium into the
policy-rate signal that swaps and FRAs, referencing an unsecured or collateralized interbank rate, mostly
avoid. A future iteration could add bonds as a cross-check where OIS liquidity is thin.

### 2. Curve construction: discount factors, not raw rates

Every quoted instrument is treated as a single-payment ("bullet") par instrument out to its maturity:

$$ DF(\text{spot}, T) = \frac{1}{1 + S(T)\cdot\alpha(T)} $$

where $S(T)$ is the quoted par rate, $\alpha(T)$ is the day-count fraction (ACT/360 or ACT/365, by
market), and $DF$ is the discount factor. Interpolation is then applied to **log discount factors**, never
to raw rates — Gordian's notes on this project flagged that linear interpolation on a concave curve
underestimates forward rates, and log-DF interpolation is the standard fix: because discount factors are
strictly positive and (for normal, upward- or downward-sloping curves) log-convex/concave in a
well-behaved way, interpolating their logarithm keeps every implied forward rate consistent with
no-arbitrage rather than producing the sawtooth or negative-forward artifacts that plain linear
rate-interpolation can create. By default this notebook uses **PCHIP** (shape-preserving cubic Hermite
interpolation), which stays close to Gordian's "exponential spline" suggestion — smooth, without the
overshoot a plain cubic spline can introduce between widely spaced nodes. A plain **linear-on-log-DF**
mode is also exposed (Assumptions panel) so users can see exactly how much the smoothing choice moves the
answer.

### 3. Poland's FRA bootstrap

A supplied WIBOR FRA strip is chained into discount factors leg by leg:

$$ DF(0, T_{i}) = DF(0, T_{i-1}) \Big/ \big(1 + \text{FRA}_{i}\cdot\alpha_i\big) $$

and then each cumulative $DF(0,T_i)$ is converted back into an equivalent bullet par rate using the same
formula as every other market, so it plugs into the identical curve/calibration engine. This is a
simplification of real WIBOR FRA settlement conventions (confirm with the desk before production use) but
uses real, supplied market levels rather than an invented curve.

### 4. Meeting-date calibration — the core of this tool

This is the direct answer to "price per central bank meeting date, not just fixed horizons." The policy
path is modeled as **piecewise-constant**, jumping only on scheduled meeting dates. Given the meeting
dates and the smooth market discount curve above, the model solves for the set of meeting-date jumps
$\{\Delta_1, \Delta_2, \dots, \Delta_n\}$ (in bp) that best reprices every calibration-tenor instrument
under that piecewise path, by nonlinear least squares:

$$ \min_{\Delta_1,\dots,\Delta_n} \sum_{k} w_k \big(\hat S_k(\Delta) - S_k^{\text{mkt}}\big)^2
   \;+\; \lambda_{\text{smooth}}\sum_i (\Delta_{i+1}-\Delta_i)^2
   \;+\; \lambda_{\text{size}}\sum_i \Delta_i^2 $$

where $\hat S_k(\Delta)$ is the model's repriced par rate for instrument $k$ under the candidate meeting
path, and the smoothness/size terms are small regularization penalties that keep the fit well-posed
whenever a single quoted tenor spans more than one meeting (there are then more meetings than
instruments, so the system is under-determined without some tie-breaking rule). Two optional penalties —
front-loaded and back-loaded — let a user express a *methodological* prior about when, within an
under-determined bucket, the market's implied move most likely lands; they change how the fit allocates
an already-implied total move across meetings, not how much total move the curve implies overall. This
approach was chosen deliberately over the simpler "N-day forward rate after each meeting" method used in
early prototypes in this project, because a fixed forward window quietly mis-prices any meeting that isn't
exactly one instrument-tenor apart from the next — exactly the kind of hidden assumption MIPR is being
built to fix.

### 5. Reading the chart and table (WIRP-style)

- **Blue step line** — the market-implied policy path: it is flat between meetings and jumps only on a
  meeting date, by construction.
- **Orange/blue bars** — "# of standard moves priced", i.e. $\Delta_i / \text{standard\_move\_bps}$ for
  that market. This is exactly Bloomberg WIRP's convention: **+0.20** reads as "the market has about 20%
  of a 25bp hike priced in", **-0.66** reads as "about 66% of a 25bp cut priced in". Orange bars are hikes,
  blue bars are cuts.
- **Table** — the same numbers in bp and rate-level terms per meeting, plus a Hike / Cut / Hold label from
  the sign of the fitted move.

### 6. Scenario entry and fair value

Users overwrite the market-implied move at each meeting with their own view (or a preset: Hold, Dovish,
Hawkish, or the market path itself as a starting point). The scenario path is then repriced into a
**par rate** for the selected trade tenor using the identical piecewise-discounting formula as the
market-implied path (never a simple average of the entered moves), and compared against the live market
rate for that same tenor:

$$ \text{gap (bp)} = 100\times\big(S^{\text{scenario}}_{\text{tenor}} - S^{\text{market}}_{\text{tenor}}\big) $$

DV01 and scenario P&L for a specified notional and Pay-fixed/Receive-fixed direction are computed from
that same discounting, so "fair value" here always means *fully discount-factor consistent* repricing
under your policy path — not a shortcut average.

### 7. Assumptions you can change

| Assumption | Where | Notes |
|---|---|---|
| Policy rate / effective O/N | Header controls | Auto-fills from Live BQL when available |
| O/N-to-policy basis (bp) | Header controls | Some markets quote an O/N reference distinct from the policy rate |
| Standard move size (bp) | Header controls | Defines the "# of moves priced" units |
| Curve interpolation | Assumptions panel | PCHIP (default) vs linear, both on log discount factors |
| Meeting allocation | Assumptions panel | Balanced / Smooth / Front-loaded / Back-loaded regularization |
| Data source | Top controls | Snapshot (offline, illustrative) vs Live BQL |
| Meeting calendar source | Top controls | Static list vs Live BQL central-bank calendar |
| Policy scenario | Scenario section | Per-meeting bp moves, or a preset |
| Trade tenor / direction / notional | Fair value section | Drives the DV01 / P&L output |

### 8. Data confidence and known limitations

Each country's Assumptions panel states its own **data confidence** and **calendar confidence**. Mexico,
Chile, Turkey, Colombia and Hungary carry desk-supplied tickers and curve snapshots. Poland's snapshot is
bootstrapped from a real FRA strip but its live curve object number is unconfirmed. India, Indonesia,
Thailand, Malaysia and the Philippines run on Live-BQL-only curve objects and policy tickers identified
from Bloomberg's standard numbering convention and public reporting, not desk confirmation — treat their
output as a working prototype and confirm before trading off it. Africa is not configured at all in this
build (no tickers or calendar data existed in the source material); adding a market only requires one new
entry in the `COUNTRIES` dict, since the curve engine, calibration engine and dashboard are all
country-agnostic.

"""


In [ ]:
# =============================================================================
# 8. LAUNCH
# =============================================================================

dashboard = GlobalEMPolicyPricer()
dashboard.display()
dashboard.run(refresh=True)


# Methodology & glossary (full write-up)

*This is the same content shown inline in the app under "Details -> Methodology & glossary" —
reproduced here as a standalone, printable reference, Bloomberg-help-page style.*

### What this replaces, and why

Bloomberg MIPR shows an implied policy rate per country at fixed horizons (e.g. "Brazil 1y implied = x%")
without disclosing which instruments it used, which interpolation, which day count, whether policy is
assumed to move only at meetings, or how the first fixing is handled. This tool makes every one of those
choices explicit and editable, and prices to actual **meeting dates** rather than fixed horizons.

### 1. Instrument choice per market

EM short-rate curves are swap/OIS-based almost everywhere in this tool's scope, because that is what is
liquid: Chile has no liquid short-dated futures or FRAs, so its curve is built entirely from Camara OIS;
Mexico trades on the TIIE-F RFR swap curve; Turkey, Colombia and Hungary similarly price off OIS/IRS
curves. Poland is the one exception in this build: no OIS curve was supplied, so its snapshot is
bootstrapped from a quoted WIBOR **FRA** strip (3x6, 6x9, ... 21x24) — see "Poland's FRA bootstrap" below.
Bonds were not used anywhere in this build: government bond yields mix a credit/liquidity premium into the
policy-rate signal that swaps and FRAs, referencing an unsecured or collateralized interbank rate, mostly
avoid. A future iteration could add bonds as a cross-check where OIS liquidity is thin.

### 2. Curve construction: discount factors, not raw rates

Every quoted instrument is treated as a single-payment ("bullet") par instrument out to its maturity:

$$ DF(\text{spot}, T) = \frac{1}{1 + S(T)\cdot\alpha(T)} $$

where $S(T)$ is the quoted par rate, $\alpha(T)$ is the day-count fraction (ACT/360 or ACT/365, by
market), and $DF$ is the discount factor. Interpolation is then applied to **log discount factors**, never
to raw rates — Gordian's notes on this project flagged that linear interpolation on a concave curve
underestimates forward rates, and log-DF interpolation is the standard fix: because discount factors are
strictly positive and (for normal, upward- or downward-sloping curves) log-convex/concave in a
well-behaved way, interpolating their logarithm keeps every implied forward rate consistent with
no-arbitrage rather than producing the sawtooth or negative-forward artifacts that plain linear
rate-interpolation can create. By default this notebook uses **PCHIP** (shape-preserving cubic Hermite
interpolation), which stays close to Gordian's "exponential spline" suggestion — smooth, without the
overshoot a plain cubic spline can introduce between widely spaced nodes. A plain **linear-on-log-DF**
mode is also exposed (Assumptions panel) so users can see exactly how much the smoothing choice moves the
answer. The chart in the next cell makes this concrete on a synthetic concave curve.

### 3. Poland's FRA bootstrap

A supplied WIBOR FRA strip is chained into discount factors leg by leg:

$$ DF(0, T_{i}) = DF(0, T_{i-1}) \Big/ \big(1 + \text{FRA}_{i}\cdot\alpha_i\big) $$

and then each cumulative $DF(0,T_i)$ is converted back into an equivalent bullet par rate using the same
formula as every other market, so it plugs into the identical curve/calibration engine. This is a
simplification of real WIBOR FRA settlement conventions (confirm with the desk before production use) but
uses real, supplied market levels rather than an invented curve.

### 4. Meeting-date calibration — the core of this tool

This is the direct answer to "price per central bank meeting date, not just fixed horizons." The policy
path is modeled as **piecewise-constant**, jumping only on scheduled meeting dates. Given the meeting
dates and the smooth market discount curve above, the model solves for the set of meeting-date jumps
$\{\Delta_1, \Delta_2, \dots, \Delta_n\}$ (in bp) that best reprices every calibration-tenor instrument
under that piecewise path, by nonlinear least squares:

$$ \min_{\Delta_1,\dots,\Delta_n} \sum_{k} w_k \big(\hat S_k(\Delta) - S_k^{\text{mkt}}\big)^2
   \;+\; \lambda_{\text{smooth}}\sum_i (\Delta_{i+1}-\Delta_i)^2
   \;+\; \lambda_{\text{size}}\sum_i \Delta_i^2 $$

where $\hat S_k(\Delta)$ is the model's repriced par rate for instrument $k$ under the candidate meeting
path, and the smoothness/size terms are small regularization penalties that keep the fit well-posed
whenever a single quoted tenor spans more than one meeting (there are then more meetings than
instruments, so the system is under-determined without some tie-breaking rule). Two optional penalties —
front-loaded and back-loaded — let a user express a *methodological* prior about when, within an
under-determined bucket, the market's implied move most likely lands; they change how the fit allocates
an already-implied total move across meetings, not how much total move the curve implies overall. This
approach was chosen deliberately over the simpler "N-day forward rate after each meeting" method used in
early prototypes in this project, because a fixed forward window quietly mis-prices any meeting that isn't
exactly one instrument-tenor apart from the next — exactly the kind of hidden assumption MIPR is being
built to fix.

### 5. Reading the chart and table (WIRP-style)

- **Blue step line** — the market-implied policy path: it is flat between meetings and jumps only on a
  meeting date, by construction.
- **Orange/blue bars** — "# of standard moves priced", i.e. $\Delta_i / \text{standard\_move\_bps}$ for
  that market. This is exactly Bloomberg WIRP's convention: **+0.20** reads as "the market has about 20%
  of a 25bp hike priced in", **-0.66** reads as "about 66% of a 25bp cut priced in". Orange bars are hikes,
  blue bars are cuts.
- **Table** — the same numbers in bp and rate-level terms per meeting, plus a Hike / Cut / Hold label from
  the sign of the fitted move.

### 6. Scenario entry and fair value

Users overwrite the market-implied move at each meeting with their own view (or a preset: Hold, Dovish,
Hawkish, or the market path itself as a starting point). The scenario path is then repriced into a
**par rate** for the selected trade tenor using the identical piecewise-discounting formula as the
market-implied path (never a simple average of the entered moves), and compared against the live market
rate for that same tenor:

$$ \text{gap (bp)} = 100\times\big(S^{\text{scenario}}_{\text{tenor}} - S^{\text{market}}_{\text{tenor}}\big) $$

DV01 and scenario P&L for a specified notional and Pay-fixed/Receive-fixed direction are computed from
that same discounting, so "fair value" here always means *fully discount-factor consistent* repricing
under your policy path — not a shortcut average.

### 7. Assumptions you can change

| Assumption | Where | Notes |
|---|---|---|
| Policy rate / effective O/N | Header controls | Auto-fills from Live BQL when available |
| O/N-to-policy basis (bp) | Header controls | Some markets quote an O/N reference distinct from the policy rate |
| Standard move size (bp) | Header controls | Defines the "# of moves priced" units |
| Curve interpolation | Assumptions panel | PCHIP (default) vs linear, both on log discount factors |
| Meeting allocation | Assumptions panel | Balanced / Smooth / Front-loaded / Back-loaded regularization |
| Data source | Top controls | Snapshot (offline, illustrative) vs Live BQL |
| Meeting calendar source | Top controls | Static list vs Live BQL central-bank calendar |
| Policy scenario | Scenario section | Per-meeting bp moves, or a preset |
| Trade tenor / direction / notional | Fair value section | Drives the DV01 / P&L output |

### 8. Data confidence and known limitations

Each country's Assumptions panel states its own **data confidence** and **calendar confidence**. Mexico,
Chile, Turkey, Colombia and Hungary carry desk-supplied tickers and curve snapshots. Poland's snapshot is
bootstrapped from a real FRA strip but its live curve object number is unconfirmed. India, Indonesia,
Thailand, Malaysia and the Philippines run on Live-BQL-only curve objects and policy tickers identified
from Bloomberg's standard numbering convention and public reporting, not desk confirmation — treat their
output as a working prototype and confirm before trading off it. Africa is not configured at all in this
build (no tickers or calendar data existed in the source material); adding a market only requires one new
entry in the `COUNTRIES` dict, since the curve engine, calibration engine and dashboard are all
country-agnostic.


In [ ]:
# =============================================================================
# 9. ILLUSTRATION -- why interpolate log discount factors, not raw rates
# =============================================================================
# A small synthetic, deliberately concave curve to make section 2 concrete:
# linear interpolation on raw par rates understates the forward rate implied
# between two widely-spaced, sharply concave nodes, while PCHIP-on-log-DF
# does not. This cell is illustrative only -- it does not use any live data.

_illustration_days = np.array([1, 90, 365])
_illustration_rates = np.array([4.50, 4.10, 3.20])  # steep, concave cut cycle

_dense_days = np.linspace(1, 365, 200)

# (a) naive: linear interpolation directly on the par rate, then read off a
#     90-365 day forward from those interpolated levels.
_linear_rate_fn = interp1d(_illustration_days, _illustration_rates, fill_value="extrapolate")
_linear_rates_dense = _linear_rate_fn(_dense_days)

# (b) this notebook's approach: PCHIP on log discount factors.
_illustration_df = 1.0 / (1.0 + _illustration_rates / 100.0 * _illustration_days / 360.0)
_illustration_logdf_fn = PchipInterpolator(_illustration_days, np.log(_illustration_df), extrapolate=True)
_pchip_df_dense = np.exp(_illustration_logdf_fn(_dense_days))
_pchip_rates_dense = (1.0 / _pchip_df_dense - 1.0) / (_dense_days / 360.0) * 100.0

_fig_illustration = go.Figure()
_fig_illustration.add_trace(go.Scatter(
    x=_illustration_days, y=_illustration_rates, mode="markers", name="Quoted nodes",
    marker={"size": 11, "color": "#f5f5f5", "symbol": "diamond"},
))
_fig_illustration.add_trace(go.Scatter(
    x=_dense_days, y=_linear_rates_dense, mode="lines", name="Linear on raw rate (naive)",
    line={"width": 2, "dash": "dot", "color": "#f87171"},
))
_fig_illustration.add_trace(go.Scatter(
    x=_dense_days, y=_pchip_rates_dense, mode="lines", name="PCHIP on log discount factor (this notebook)",
    line={"width": 3, "color": "#38bdf8"},
))
_fig_illustration.update_layout(
    title="Illustration only (synthetic curve): why this notebook interpolates log discount factors",
    xaxis_title="Days", yaxis_title="Rate (%)",
    template="plotly_dark", height=420, hovermode="x unified",
    legend={"orientation": "h", "yanchor": "bottom", "y": 1.02, "x": 0.0},
)
display(_fig_illustration)
display(HTML(
    "<div style='color:#9ca3af;font-size:12px;'>On a sharply concave curve like this one, naive linear "
    "interpolation on raw rates cuts straight through the middle node and understates the curvature; the "
    "log-discount-factor / PCHIP approach used throughout this notebook tracks it. This is a synthetic "
    "example for illustration, not live market data.</div>"
))
